# Exploratory Data Analysis — Customer Shopping Behavior

This notebook performs a structured EDA of `customer_shopping_behavior-clean.csv`.

### Sections
1. Load the data
2. Dataset overview
3. Data quality and missing values
4. Descriptive statistics
5. Univariate analysis
6. Categorical analysis
7. Numerical relationships
8. Customer and purchase behavior
9. Time-based analysis
10. Key EDA findings


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

CSV_PATH = "/mnt/data/customer_shopping_behavior-clean.csv"

df = pd.read_csv(CSV_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())


## 1. Dataset Overview

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nRandom sample:")
display(df.sample(min(10, len(df)), random_state=42))


## 2. Data Quality

In [ ]:
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100),
    "unique_values": df.nunique(dropna=True)
}).sort_values("missing_count", ascending=False)

display(quality)

print("Duplicate rows:", df.duplicated().sum())


## 3. Descriptive Statistics

In [ ]:
display(df.describe(include="all").T)

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)


## 4. Numerical Distributions

In [ ]:
for col in numeric_cols:
    plt.figure(figsize=(8, 4))
    plt.hist(df[col].dropna(), bins=30)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


## 5. Categorical Variables

In [ ]:
for col in categorical_cols:
    counts = df[col].value_counts(dropna=False).head(15)

    plt.figure(figsize=(9, 5))
    counts.sort_values().plot(kind="barh")
    plt.title(f"Top Categories — {col}")
    plt.xlabel("Count")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

    display(counts.to_frame("count"))


## 6. Correlation Analysis

In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    display(corr)

    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.index)), corr.index)
    plt.title("Numerical Correlation Matrix")
    plt.tight_layout()
    plt.show()


## 7. Customer Behavior

In [ ]:
if "Customer ID" in df.columns:
    customer_summary = df.groupby("Customer ID").agg(
        purchases=("Customer ID", "size")
    ).sort_values("purchases", ascending=False)

    display(customer_summary.describe())
    display(customer_summary.head(20))

    plt.figure(figsize=(8, 4))
    plt.hist(customer_summary["purchases"], bins=30)
    plt.title("Purchases per Customer")
    plt.xlabel("Number of purchases")
    plt.ylabel("Number of customers")
    plt.tight_layout()
    plt.show()


## 8. Product Analysis

In [ ]:
if "Item Purchased" in df.columns:
    product_counts = df["Item Purchased"].value_counts()

    print("Number of unique products:", df["Item Purchased"].nunique())
    display(product_counts.head(20).to_frame("purchases"))

    plt.figure(figsize=(10, 6))
    product_counts.head(15).sort_values().plot(kind="barh")
    plt.title("Top 15 Most Purchased Items")
    plt.xlabel("Purchases")
    plt.ylabel("Item")
    plt.tight_layout()
    plt.show()


## 9. Location Analysis

In [ ]:
if "Location" in df.columns:
    location_counts = df["Location"].value_counts()

    print("Number of locations:", df["Location"].nunique())
    display(location_counts.head(20).to_frame("purchases"))

    plt.figure(figsize=(10, 6))
    location_counts.head(15).sort_values().plot(kind="barh")
    plt.title("Top Locations by Number of Purchases")
    plt.xlabel("Purchases")
    plt.ylabel("Location")
    plt.tight_layout()
    plt.show()


## 10. Purchase Date Analysis

In [ ]:
if "Purchase Date" in df.columns:
    df["Purchase Date"] = pd.to_datetime(df["Purchase Date"], errors="coerce")

    print("Date range:")
    print(df["Purchase Date"].min(), "to", df["Purchase Date"].max())

    monthly = (
        df.dropna(subset=["Purchase Date"])
          .set_index("Purchase Date")
          .resample("M")
          .size()
    )

    plt.figure(figsize=(11, 4))
    monthly.plot()
    plt.title("Purchases Over Time")
    plt.xlabel("Month")
    plt.ylabel("Number of purchases")
    plt.tight_layout()
    plt.show()


## 11. Purchase Behavior by Month and Day of Week

In [ ]:
if "Purchase Date" in df.columns:
    temp = df.dropna(subset=["Purchase Date"]).copy()
    temp["month"] = temp["Purchase Date"].dt.month
    temp["day_of_week"] = temp["Purchase Date"].dt.day_name()

    month_counts = temp["month"].value_counts().sort_index()
    dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday",
                 "Friday", "Saturday", "Sunday"]
    dow_counts = temp["day_of_week"].value_counts().reindex(dow_order)

    fig, ax = plt.subplots(figsize=(8, 4))
    month_counts.plot(kind="bar", ax=ax)
    ax.set_title("Purchases by Month")
    ax.set_xlabel("Month")
    ax.set_ylabel("Purchases")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 4))
    dow_counts.plot(kind="bar", ax=ax)
    ax.set_title("Purchases by Day of Week")
    ax.set_xlabel("Day")
    ax.set_ylabel("Purchases")
    plt.tight_layout()
    plt.show()


## 12. Cross-Tabulations

In [ ]:
# Useful categorical cross-tabs when these columns exist
if {"Location", "Item Purchased"}.issubset(df.columns):
    location_item = pd.crosstab(
        df["Location"], df["Item Purchased"]
    )

    print("Location × Item matrix shape:", location_item.shape)
    display(location_item.head())

if {"Gender", "Item Purchased"}.issubset(df.columns):
    gender_item = pd.crosstab(
        df["Gender"], df["Item Purchased"],
        normalize="index"
    ).mul(100)

    display(gender_item.round(2))


## 13. Automated EDA Summary

In [ ]:
print("=" * 70)
print("EDA SUMMARY")
print("=" * 70)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Duplicate rows: {df.duplicated().sum():,}")
print(f"Missing cells: {int(df.isna().sum().sum()):,}")

if "Customer ID" in df.columns:
    print(f"Unique customers: {df['Customer ID'].nunique():,}")

if "Item Purchased" in df.columns:
    print(f"Unique items: {df['Item Purchased'].nunique():,}")
    print(f"Most purchased item: {df['Item Purchased'].value_counts().idxmax()}")

if "Location" in df.columns:
    print(f"Unique locations: {df['Location'].nunique():,}")
    print(f"Most active location: {df['Location'].value_counts().idxmax()}")

if "Purchase Date" in df.columns:
    print(f"Purchase period: {df['Purchase Date'].min()} → {df['Purchase Date'].max()}")

print("\nTop missing-value columns:")
display(
    df.isna().sum()
      .sort_values(ascending=False)
      .head(10)
      .to_frame("missing_count")
)
